# 4-3 이진 분류 출력 심화

강의 원문 대신 직접 작성하고 실행한 코드와 학습 메모를 정리했습니다.


In [1]:
# 검증 가능 정답 코드
import torch
from torch import nn
logits = torch.tensor([[0.0], [2.0]])
# BCEWithLogitsLoss의 target은 logit과 같은 (B,1) shape의 float 계약을 사용합니다.
target = torch.tensor([[0.0], [1.0]], dtype=torch.float32)
loss_fn = nn.BCEWithLogitsLoss()
# 정답 경로는 raw logits를 전달하고, wrong 경로는 Sigmoid를 미리 적용해 변환을 중복시킵니다.
correct = loss_fn(logits, target)
wrong = loss_fn(torch.sigmoid(logits), target)
print("losses:", f"{correct.item():.4f}", f"{wrong.item():.4f}")
print("same:", torch.allclose(correct, wrong))

losses: 0.4100 0.6604
same: False


In [5]:
# 검증 가능 정답 코드
import torch

def binary_infer(logits, threshold=0.5):
    # 샘플당 logit 하나라는 이진 출력 계약과 확률 threshold 범위를 추론 경계에서 실패시킵니다.
    assert logits.ndim == 2 and logits.shape[1] == 1, "expected (B,1) logits"
    assert 0.0 <= threshold <= 1.0, "probability threshold out of range"
    # threshold는 raw logit이 아니라 Sigmoid로 변환한 확률 공간에서 적용합니다.
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).long()
    return probs, preds

logits = torch.tensor([[-1.0], [0.0], [1.0], [2.0]])
probs, preds = binary_infer(logits, 0.7)
print("probs:", [round(v, 4) for v in probs.squeeze(1).tolist()])
print("preds:", preds.squeeze(1).tolist())

probs: [0.2689, 0.5, 0.7311, 0.8808]
preds: [0, 0, 1, 1]


In [6]:
# 검증 가능 정답 코드
import torch
probs = torch.tensor([0.90, 0.65, 0.55, 0.40])
y = torch.tensor([1, 0, 1, 0])
costs = {}
# 같은 validation 표본에서 FN 비용 5와 FP 비용 2를 적용해 threshold별 업무 비용을 복원합니다.
for threshold in (0.5, 0.7):
    pred = (probs >= threshold).long()
    fn = int(((pred == 0) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    costs[threshold] = 5 * fn + 2 * fp
# 정확도나 threshold 크기가 아니라 사전에 합의한 총비용이 가장 낮은 후보를 선택합니다.
selected = min(costs, key=costs.get)
print("costs:", costs)
print("selected:", selected)

costs: {0.5: 2, 0.7: 5}
selected: 0.5
